# Project Bishop: automated quality checks

This notebook runs the automated part of the quality system for the Bishop v2.0 guidelines
(egocentric video, segmentation then captioning). The code lives in the `bishop_qc` package
in this repository. The notebook imports it, runs it on the sample recordings and shows each
step.

| Section | What it does |
|---|---|
| 1 | Setup, device and API keys |
| 2 | Input data and the data contract |
| 3 | Other assumptions |
| 4 | Shared types, and the caption parser's literals, regex and mappings |
| 5 | Primitives: computed once, added to the segments CSV |
| 6 | Demo inputs: videos and three sample segments |
| 7 | Primitive outputs |
| 8 | Checks, category by category. Done so far: Segmentation errors (all three categories) and Caption errors (Faithfulness, Completeness, Hands and body attribution, Navigation, Object reference) |

**To run in Colab**
1. Open Runtime > Change runtime type and pick a GPU.
2. Add these Colab secrets: `OPENAI_API_KEY` (Sol and OpenAI Decisions), `JEV_API_KEY` (Jev) and
   `HF_TOKEN` (SAM 3 is a gated model on Hugging Face). Locally, the same names in `.env.local`
   at the repository root are read instead. Values are never printed.
3. The first cell clones https://github.com/AvinaashAnandK/EncordSubmission. Set `BISHOP_REPO_URL` to use a fork.
4. Run all cells. The sample videos are not in git, because GitHub rejects files over 100 MB.
   Section 2 downloads any missing video from a public Google Drive folder and checks it against
   `SampleInput/videos.sha256`, so the checks always see the original frames.
5. The heavy primitives cache to `cache/` and `outputs/`, so a rerun is fast.

Clear the cell outputs before you commit. The embedded videos make the file large.

## 1. Setup and keys

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
BISHOP_REPO_URL = os.getenv("BISHOP_REPO_URL", "https://github.com/AvinaashAnandK/EncordSubmission.git")

if IN_COLAB:
    import importlib.util
    root = Path("/content/bishop")
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1", BISHOP_REPO_URL, str(root)], check=True)
    else:
        subprocess.run(["git", "-C", str(root), "pull", "--ff-only"], check=True)   # pick up new commits
    os.chdir(root)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    if importlib.util.find_spec("en_core_web_sm") is None:
        # The model name must come first: spaCy reads anything before it as the package name.
        subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)

ROOT = Path.cwd() if (Path.cwd() / "bishop_qc").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
print("Repository root:", ROOT)

In [ ]:
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
import json
import pandas as pd
from IPython.display import display, Markdown

from bishop_qc.config import make_config, select_device, load_keys
from bishop_qc.clients import build_clients, client_status
from bishop_qc import parser as caption_parser

pd.set_option("display.max_colwidth", 120)
DEVICE = select_device()          # CUDA first, then Apple MPS. Stops with a note when neither exists.
KEYS = load_keys(root=ROOT)       # environment, then .env.local or .env, then Colab secrets
cfg = make_config(ROOT, data_dir=os.getenv("BISHOP_DATA_DIR"))
cfg.device = DEVICE
caption_parser.set_model(cfg.spacy_model)
clients = build_clients(cfg)

if KEYS.get("HF_TOKEN"):
    from huggingface_hub import login
    login(os.environ["HF_TOKEN"], add_to_git_credential=False)

print("Device:", DEVICE)
display(pd.DataFrame([{"key": k, "set": v} for k, v in KEYS.items()]))
display(pd.DataFrame(client_status(clients)))

## 2. Input data and assumptions

**Recordings** have an `recording_id` and a `path` to the video.

**Segments** have the fields below. The sample CSVs hold only the index, the times and the
caption. The platform fields are assumed to be obtainable. For the demo they are filled with
seeded synthetic values, marked in the `synthetic_fields` column, with a few planted cases so the
platform checks have something to find.

The loader reads `recordings.csv` and `segments.csv` from the data folder when they exist.
Otherwise it pairs each `<Name>.mp4` with `<name>_annotations.csv`.

The CSVs are in git. The videos are not (GitHub rejects files over 100 MB). Any video missing from the
data folder is downloaded from the public Google Drive folder in `cfg.videos_drive_folder`, then every
video is checked against `SampleInput/videos.sha256`. A mismatch stops the run, because the checks must
see the frames the annotators saw.

In [ ]:
from bishop_qc.schema import RECORDING_COLUMNS, SEGMENT_COLUMNS, PLATFORM_COLUMNS

in_sample = {"recording_id", "segment_id", "segment_index", "start_s", "end_s", "caption"}
contract = pd.DataFrame(
    [{"table": "recordings", "field": k, "meaning": v, "in sample data": "yes"} for k, v in RECORDING_COLUMNS.items()]
    + [{"table": "segments", "field": k, "meaning": v,
        "in sample data": "yes" if k in in_sample else "no, synthetic for the demo"} for k, v in SEGMENT_COLUMNS.items()])
contract

In [ ]:
from bishop_qc import data

display(data.ensure_videos(cfg.data_dir, cfg.videos_drive_folder))
recordings = data.discover_recordings(cfg.data_dir)
segments = data.load_segments(recordings, cfg.data_dir)
if cfg.synthesize_platform_fields:
    segments, plants = data.add_synthetic_platform_fields(segments, cfg.seed)

problems = data.validate_segments(segments)
print(f"{len(recordings)} recordings, {len(segments)} segments, {len(problems)} contract problems")
display(recordings)
display(segments.head(6))
display(Markdown("**Planted cases in the synthetic fields**"))
display(plants)

## 3. Other assumptions

**Tool and timing**
1. The annotation tool enforces no gaps, no overlaps and the 0.2 s minimum. The export gate still
   checks all three, because the segmentation key bindings are not final.
2. The 0.1 s boundary tolerance is turned into frames per video from the measured frame
   timestamps, not the nominal frame rate. The samples run at 50 fps (0.1 s = 5 frames) and
   23.976 fps (0.1 s = 2.4 frames).
3. The CSV stores times to 0.1 s. Rounding adds up to 0.05 s, so the boundary checks use 0.15 s.
4. Timelines are sampled at 10 fps. Every frame within 0.5 s of a boundary is kept at the native rate.

**Models**
5. Hands: the hand detector from WiLoR (YOLOv8, classes left and right) gives the box and the side
   of each hand. Its license is CC BY-NC 4.0, which suits this demo but not commercial delivery.
6. **Alternative for hands and contact: EgoHOS** (Zhang et al., ECCV 2022, MIT license). It segments
   the left hand, the right hand and the object each hand touches, trained on egocentric video, so it
   gives side and contact from one model. It is not on Hugging Face (weights on Google Drive) and it
   needs torch 1.11 and mmcv-full 1.6 in a separate Python 3.10 environment, so it would run as its
   own step that writes masks to the cache. See https://github.com/owenzlz/EgoHOS.
7. The hand detector has no ego or other person label. A box that reaches the bottom edge of the
   frame is marked `ego`. Any other box is `unclear`, and only routes the segment to review.
8. Contact is a heuristic: the share of the SAM 3 hand mask, grown by 6 px, that touches a SAM 3
   mask of a caption noun. It must be calibrated on the gold set.
9. SAM 3 is prompted only with the nouns of the segment, its neighbors, "hand" and "person". A
   missed detection does not prove an object is out of view.
10. Camera motion comes from an affine fit to RAFT optical flow. The values are relative. None is
    an angle in degrees.
11. Captions are parsed with rules plus spaCy. Sol extracts the fields only when the rules leave a
    warning. All 135 sample captions parse with rules alone.
12. API models. Sol (`gpt-6.1-sol`) takes text and images but no video, so a clip is sent as
    frames. OpenAI Decisions (`/v1/decisions` with `gpt-6-luna`) answers closed questions on images.
    It has no public reference, so its request format was confirmed against the live API on
    2026-10-15. It falls back to Sol on any failure. Jev answers closed questions on text through
    TypeSafe's API. The alias `jev-latest` currently resolves to `jev-1.13.0`, not 1.14.
13. A missing key, a model error or an unreadable answer gives `unassessable`. It never gives a pass.

**Platform fields**
14. `accepted_type` is `accepted_without_edits` or `accepted_with_edits`. A caption written from
    scratch would need a third value. The loader reports any unknown value.
15. `caption_created_at` is the first commit of the caption and `caption_updated_at` the last edit,
    both in UTC.
16. A recording can be split across annotators. Per annotator rates use `annotator_id`.

**Cost and time**
17. Every primitive and every check records its wall time per segment. Work done once per
    recording is split across segments by frame count.
18. Prices per 1M tokens. Sol: USD 2 input, 0.10 cached, 10 output (OpenAI's announcement).
    gpt-6-luna: USD 0.10 input, 0.50 output. Jev: USD 0.042 input, output free. The last two come
    from third-party price trackers, not the providers' own pages. A cached call costs nothing in
    the run, and the report also keeps what it costs uncached.
19. Compute cost uses an ASSUMED rate of USD 0.50 per GPU hour and 0.05 per CPU hour
    (`cfg.gpu_usd_per_hour`, `cfg.cpu_usd_per_hour`). Replace them with the real rates.

**Scope**
20. Originals are never changed. Every output goes to `outputs/`. A proposed fix is a separate column.
21. No gold set exists yet. Every threshold is a placeholder until it is calibrated on one.

## 4. Shared types and caption parser literals, regex and mappings

**Types.** A `Clause` is one action of a caption. A `ParsedCaption` holds its clauses. An `Answer`
is one API model answer, and a `Finding` is one check result on one segment.

In [ ]:
import dataclasses
from bishop_qc.schema import Clause, ParsedCaption, Answer, Finding

def fields_table(cls):
    return pd.DataFrame([{"type": cls.__name__, "field": f.name,
                          "python type": str(f.type).replace("typing.", "")} for f in dataclasses.fields(cls)])

pd.concat([fields_table(c) for c in (Clause, ParsedCaption, Answer, Finding)], ignore_index=True)

**Literals, word lists, regex and mappings.** Lists marked CLIENT in `bishop_qc/lexicon.py` need sign
off before a check that uses them counts in a score.

In [ ]:
from bishop_qc import lexicon as L

def preview(v, n=8):
    if hasattr(v, "pattern"):
        return v.pattern[:110] + ("..." if len(v.pattern) > 110 else "")
    if isinstance(v, dict):
        items = [f"{k} -> {x}" for k, x in list(v.items())[:n]]
    else:
        items = [" / ".join(sorted(x)) if isinstance(x, frozenset) else str(x) for x in list(v)[:n]]
    return ", ".join(items) + (" ..." if len(v) > n else "")

rows = []
for name in sorted(dir(L)):
    v = getattr(L, name)
    if name.isupper() and not callable(v):
        kind = "regex" if hasattr(v, "pattern") else type(v).__name__
        rows.append({"name": name, "kind": kind, "size": "" if kind == "regex" else len(v), "values": preview(v)})
pd.DataFrame(rows)

**Parser.** The parser splits a caption into clauses on `then`, `but`, `while`, and on `and` when a verb
follows. For each clause it reads the verb and particle, the actor (hand or body part), the object with
its attributes, the place, the landmarks, the stopping point and any held object. It parses the text as
"You ..." so spaCy reads the imperative verb as a verb. The guideline's own examples below are the test
cases.

In [ ]:
examples = [
    "Pick up the cup from the table with your right hand",
    "Walk past the kitchen island toward the sink while holding the cup in your right hand",
    "Turn right until you are facing the sink",
    "Attempt to grasp the jar with your right hand but miss the jar",
    "Repeatedly pinch the white towel with your right hand then align its bottom-right corner with the other towel using both hands",
    "Push the cabinet door closed with your hip while holding the stack of plates in both hands",
    "Twist the cap onto the bottle with your right hand until it is closed",
    "Pick up the gray garment from the couch with your left hand",
    "IDLE",
]
show = ["connector", "intent", "adverb", "verb_key", "object", "attributes", "actor", "tool", "place_prep",
        "place", "place_from", "landmarks", "direction", "facing", "stop", "held_object", "held_actor",
        "is_navigation", "has_failure_word"]
rows = []
for cap in examples:
    pc = caption_parser.parse_caption(cap)
    if not pc.clauses:
        rows.append({"caption": cap, "source": pc.source, "literal": pc.literal})
    for c in pc.clauses:
        d = c.to_dict()
        rows.append({"caption": cap if c.index == 0 else "", "source": pc.source, **{k: d[k] for k in show}})
pd.DataFrame(rows).fillna("")

## 5. Primitives

Primitives are the shared signals the checks read. Each one is computed once per recording, cached,
and its per segment summary is added as columns to `outputs/segments_enriched.csv`. Per frame data
goes to `outputs/timelines/`. Every stage records its time and cost per segment.

| | Primitive | Source | Output | Used by sheet rows |
|---|---|---|---|---|
| P1 | Video metadata | ffprobe | frame times, measured fps, 0.1 s in frames, duration, audio | 2 to 6, all frame sampling |
| P2 | Frames and blur | ffmpeg / OpenCV | 10 fps timeline frames, native frames within 0.5 s of each boundary, blur score | every frame based row, 34, 58 |
| P3 | Hand timeline | WiLoR hand detector + SAM 3 hand masks | per frame and side: box, owner (`ego` or `unclear`), state (`contact`, `no_contact`, `unknown`), contacted object | 2, 3, 8, 9, 11, 13 to 18, 24, 29, 37 to 48 |
| P4 | Motion timeline | RAFT optical flow | per frame pair: camera pan, forward movement, hand motion; onsets | 2, 3, 6, 13, 18, 20 to 22, 24, 26, 55 |
| P5 | Caption fields | rules + spaCy, Sol fallback | one record per clause | every caption row |
| P6 | Object tracks | SAM 3, text prompts from P5 | per segment and noun: boxes, masks, scores, track ids; hand to object overlap | 7, 14, 16, 29, 31, 32, 37, 43, 47, 48, 57 to 60, 63 to 65, 70 |
| P7 | Segment summary | joins P2 to P6 | contact shares and changes, found objects, motion shares, onset near the start | most checks |
| P8 | Suggestion features | suggestions, accepted_type, P5 | nearest suggestion, distance, changed fields, failure added | 29, 33, 34, 58 |
| P9 | Edit time features | created and updated times, annotator | edit lag, gap since the last caption, backward edits, repeated blocks, annotator switches | 19, 24, 65, 68, 82 |

**Hands and contact.** The WiLoR detector gives side. SAM 3 segments "hand", and each hand mask is
matched to a detector box by overlap. Contact is the share of the grown hand mask that touches an
object mask. **EgoHOS is the alternative** for this primitive: one egocentric model that segments
each hand and the object it touches, with an MIT license, at the cost of a separate legacy
environment (see assumption 6).

**Run time.** SAM 3 (P6) is the slowest primitive. Each stage prints its progress, and P3, P4 and P6 save
each recording as soon as it finishes, so an interrupted run keeps what it has done. On a T4 the next cell
switches SAM 3 to 2 frames per second at 560 px, several times faster at some cost in accuracy. On a
larger GPU (L4, A100) it keeps 5 frames per second at the native 1008 px.

In [ ]:
if DEVICE == "cuda":
    import torch
    gpu = torch.cuda.get_device_name(0)
    if "T4" in gpu:
        cfg.sam3_fps, cfg.sam3_image_size = 2.0, 560
    print(f"GPU: {gpu}. SAM 3 at {cfg.sam3_fps} frames per second, {cfg.sam3_image_size or 1008} px.")

In [ ]:
from bishop_qc.primitives import run_primitives, ALL_STAGES

prims = run_primitives(cfg, recordings, segments, clients=clients, stages=ALL_STAGES, device=DEVICE)
print(f"segments_enriched.csv: {prims.segments.shape[0]} rows, {prims.segments.shape[1]} columns")

## 6. Demo inputs

Click a row to play that segment in the player beside it.

In [ ]:
from bishop_qc import display as V

V.video_gallery(recordings, segments, cfg.cache_dir)

Three sample segments, one per recording, each with everything we have about it.

| Segment | Why it is interesting |
|---|---|
| `sample1_016` | Two hands doing different things, and an object lowered out of view |
| `sample2_004` | "Pick up ... then put ...": two actions in one segment by the guideline |
| `sample3_010` | Washes the fork with the right hand right after the left hand picked it up |

In [ ]:
DEMO = ["sample1_016", "sample2_004", "sample3_010"]
rec_by_id = prims.recordings.set_index("recording_id")
for sid in DEMO:
    row = segments[segments.segment_id == sid].iloc[0]
    display(Markdown(f"### {sid}"))
    V.segment_card(row, rec_by_id.loc[row.recording_id])

## 7. Primitive outputs

**P1 video metadata**

In [ ]:
prims.recordings

**P5 caption fields** for the demo segments, and the parse source across all captions

In [ ]:
display(prims.clauses[prims.clauses.segment_id.isin(DEMO)][["segment_id"] + show].fillna(""))
display(prims.segments.p5_source.value_counts().rename("captions"))

**P8 suggestion features and P9 edit time features**: the planted cases

In [ ]:
cols = ["segment_id", "caption", "accepted_type", "p8_nearest_text", "p8_nearest_dist", "p8_exact_match",
        "p8_accepted_type_consistent", "p8_failure_added", "p9_edit_lag_s", "p9_gap_prev_s",
        "p9_backward_edit", "p9_annotator_switch", "p9_repeat_of"]
s = prims.segments
flagged = s[(~s.p8_accepted_type_consistent.astype(bool)) | s.p8_failure_added.astype(bool) | s.p9_backward_edit.astype(bool)
            | s.p9_annotator_switch.astype(bool) | (s.p9_repeat_of.fillna("") != "")]
print(f"{len(flagged)} segments with a platform signal")
flagged[cols]

**P2 frames and blur**

In [ ]:
prims.frames.groupby("recording_id").agg(frames=("frame_idx", "size"), timeline=("is_timeline", "sum"),
                                         near_boundary=("near_boundary", "sum"),
                                         blur_median=("blur", "median"), blur_p10=("blur", lambda b: b.quantile(0.1)))

**P3 hand timeline and P4 motion timeline.** Top row: segments (gray = a fixed literal). Then the state
of each hand: dark blue is contact, light blue is a detected hand with no contact, gray is unknown.
Then the three motion signals, each with its onset threshold (dashed) and onsets (dots).

In [ ]:
V.recording_timeline(prims, "sample1")
V.recording_timeline(prims, "sample2")
V.recording_timeline(prims, "sample3", 0, 60)

**P3 and P6 on single frames**: hand boxes with side and owner, SAM 3 boxes with prompt and score

In [ ]:
import matplotlib.pyplot as plt

if len(prims.objects) or len(prims.hand_dets):
    fig, axes = plt.subplots(1, len(DEMO), figsize=(6.4 * len(DEMO), 3.8))
    for ax, sid in zip(axes, DEMO):
        f = prims.frames[(prims.frames.segment_id == sid) & prims.frames.is_timeline]
        V.frame_overlay(prims, f.recording_id.iloc[0], int(f.frame_idx.iloc[len(f) // 2]), ax=ax)
    fig.tight_layout()
    plt.show()

**P6 object tracks** for the demo segments

In [ ]:
if len(prims.objects):
    display(prims.objects[prims.objects.segment_id.isin(DEMO)]
            .groupby(["segment_id", "prompt"])
            .agg(frames=("frame_idx", "nunique"), tracks=("obj_id", "nunique"), score_max=("score", "max"))
            .round(3))

**P7 segment summary** for the demo segments

In [ ]:
p7 = [c for c in prims.segments.columns if c.startswith("p7_")]
prims.segments.set_index("segment_id").loc[DEMO, p7].T

**Time and cost of the primitives.** Per stage totals, then per segment for the demo segments.
Compute cost uses the assumed hourly rates in assumption 19.

In [ ]:
display(V.cost_summary(prims.cost_per_segment))
display(prims.cost_per_segment[prims.cost_per_segment.segment_id.isin(DEMO)]
        .pivot_table(index="segment_id", columns="name", values="seconds", aggfunc="sum").round(4))

**Files written**

| File | Content |
|---|---|
| `outputs/segments_enriched.csv` | input segments plus every `p5_` to `p9_` column |
| `outputs/recordings_enriched.csv` | recordings plus `p1_` columns |
| `outputs/caption_clauses.csv` | one row per parsed clause |
| `outputs/timelines/*.csv` | frames, hand detections, hand timeline, hand to object overlap, objects, motion, onsets |
| `outputs/costs/*.csv` | primitive time and cost events, and the per segment summary |

## 8. Checks

Each check runs on a unit: a segment, a boundary between two segments, or a recording. It returns
one `Finding` per unit with a status, a message and the evidence behind it. The runner times every
unit inside the cost ledger, so each check reports its time and USD per unit in two ways:

- **check**: the check alone, including its API calls.
- **loaded**: the check plus the primitives it reads (`uses`), counted in full. That is the cost to
  run the check on its own.

Every finding is written to `outputs/checks/<check>_findings.csv` and every cost to
`outputs/costs/<check>_cost_per_unit.csv`.

In [ ]:
from bishop_qc.checks import REGISTRY, CheckContext, run_check, evidence
from bishop_qc.schema import FINDING_STATUSES

ctx = CheckContext(cfg, prims, clients)
RESULTS = {}
VIDEO_HOURS = prims.recordings.p1_duration_s.sum() / 3600

from bishop_qc.checks.report import summary_table, segment_rollup, rollup_sentence, check_sentence

def category_summary(keys, section):
    # Pass counts per check, a per segment roll-up, then time and cost.
    res = {k: RESULTS[k] for k in keys}
    roll = segment_rollup(res, prims.segments)
    display(Markdown(f"**{rollup_sentence(roll, section)}**"))
    display(Markdown("\n".join(f"- {check_sentence(v)}" for v in res.values())))
    display(Markdown("**Results per check**"))
    display(summary_table(res))
    flagged = roll[roll.result == "flagged"].merge(prims.segments[["segment_id", "caption"]], on="segment_id")
    if len(flagged):
        display(Markdown(f"**Flagged segments and why** ({len(flagged)})"))
        display(flagged[["segment_id", "caption", "flags", "reasons"]])
    display(Markdown("**Time and cost per check** (seconds and USD per unit; totals over all units)"))
    display(pd.concat([v.cost_summary() for v in res.values()], ignore_index=True).set_index("check").round(6))
    proj = pd.DataFrame([{
        "check": k,
        "usd_per_video_hour_check": v.loaded.check_usd.sum() / VIDEO_HOURS,
        "usd_per_video_hour_loaded": v.loaded.loaded_usd.sum() / VIDEO_HOURS,
        "seconds_per_video_hour_loaded": v.loaded.loaded_seconds.sum() / VIDEO_HOURS,
    } for k, v in res.items()]).set_index("check")
    display(Markdown(f"**Projected to one hour of video** (the samples are {VIDEO_HOURS * 60:.1f} minutes)"))
    display(proj.round(4))

def show_clip(sid, title="", n=8):
    s = prims.segments.set_index("segment_id").loc[sid]
    V.clip_strip(prims, s.recording_id, s.start_s, s.end_s, n=n,
                 title=title or f'{sid} ({s.start_s:.1f} to {s.end_s:.1f} s): "{s.caption}"')

display(pd.DataFrame([{"status": k, "meaning": v} for k, v in FINDING_STATUSES.items()]))

### 8.1 Segmentation errors / Boundary placement (sheet rows 2 to 7)

A cut at the wrong frame leaks frames between neighbours. The guideline allows 0.1 s, and the CSV
rounds to 0.1 s, so the checks use 0.15 s (assumption 3). The order below runs the export gate
first, because the other checks assume a clean timeline.

#### Row 5: export gate

**Approach.** Pure rules on the times, no model. For every segment: the row is complete, the id is
unique and in order, the times are finite and inside the video, the end equals the next start
(numeric tolerance 0.001 s, not the 0.1 s annotation tolerance), the segment lasts 0.2 s or more,
the first starts at 0, and the last ends within one frame of the video end. A failure is a gate
failure. The gate does not decide whether the annotator or the tool caused it.

In [ ]:
RESULTS["row05"] = r = run_check(REGISTRY["row05"], ctx)
V.check_report(r)

#### Rows 2 and 3: one boundary event, a late or an early cut

The segments touch, so one boundary is the end of one segment and the start of the next. Row 2 (a
late cut: the next action already started before the cut) and row 3 (an early cut: the earlier
action still runs after the cut) are the same event read from two sides. The check gives one
finding per boundary.

**Approach.**
1. **Signals (P3, P4).** In a 0.5 s window around the cut, list every frame where something visibly
   starts: a motion onset (hand motion, camera turn, forward movement) or a change of hand contact on
   the acting hand. The nearest one is the trigger. A trigger more than 0.15 s before the cut is late,
   more than 0.15 s after it is early. No trigger in the window is also flagged, since a large error
   can sit outside it.
2. **Image comparison (OpenAI Decisions).** One call per boundary with two frames: the frame at the
   cut minus 0.15 s and the frame at the cut plus 0.15 s, with both captions. The question asks when
   the person switches from the first action to the second: before the first frame (late), between
   the frames (the cut is right), after the second frame (early), or unclear. A probability of 0.7 or
   more on late or early raises a flag.
3. A random 5 percent of unflagged boundaries is marked for review, to measure what the check misses.

**Why a comparison.** A first version asked one question per frame ("has the next action started
in this image?"). It flagged 66 percent of the sample boundaries. A still frame of a hand holding a
piece reads as "the put has started". The two frame comparison asks the real question, whether the
switch falls inside the tolerance, and flags 6 percent.

In [ ]:
RESULTS["row02_03"] = r = run_check(REGISTRY["row02_03"], ctx)
V.check_report(r)
display(Markdown("Detail with the evidence behind each answer:"))
e = evidence(r.findings)
display(e[e.status == "candidate"][["segment_id", "row", "message", "offset_s", "late_prob",
                                    "between_prob", "early_prob", "image_source"]])
print("Marked for the random audit:", int(e.audit_sample.sum()), "boundaries")

In [ ]:
seg_idx = prims.segments.set_index("segment_id")
for row in e[e.status == "candidate"].sort_values("early_prob", ascending=False).head(3).itertuples():
    prev, nxt = seg_idx.loc[row.prev_id], seg_idx.loc[row.segment_id]
    V.boundary_strip(prims, row.recording_id, row.t, span=0.6,
                     marks=[m for m in (row.t_before, row.t_after) if m == m],
                     title=f'{row.segment_id} at {row.t:.1f} s: "{prev.caption}" then "{nxt.caption}". {row.message}')

Frames marked `*` are the two the model compared. Blue titles are after the cut.

#### Row 4: possible filler merge

**Approach.** A merged filler (under 0.2 s) leaves no trace in the CSV. It shows only as a boundary
shift. Take the boundary flags from rows 2 and 3 with a shift between 0.1 s and 0.2 s. Tag one as a
possible filler merge only when the motion timeline shows a burst under 0.2 s at the cut. A flag is
never waived because of its size. Needs the signal offset from P4, so it is unassessable without it.

In [ ]:
RESULTS["row04"] = r = run_check(REGISTRY["row04"], ctx)
V.check_report(r)

#### Row 6: reach start vs the earliest visible evidence

**Approach.** The guideline puts the cut at the first frame where the trigger becomes visible. A
reach that starts below the view is not visible, so the reference is the earliest visible evidence:
the acting hand entering the view (P3) or its motion starting (P4), in the window from 1 s before to
0.5 s after the start. This is a measurement per segment, reported for the recording. A start that
sits after the evidence again and again shows a habit. A segment with no visible evidence is
unassessable; the check never invents a start frame.

In [ ]:
RESULTS["row06"] = r = run_check(REGISTRY["row06"], ctx)
V.check_report(r)
display(Markdown("Detail with the evidence behind each answer:"))
e6 = evidence(r.findings)
if "gap_s" in e6:
    display(e6.groupby("recording_id").gap_s.describe().round(3))

#### Row 7: a split where the same action resumes

**Approach.** For every run of three segments S1, S2, S3: match the verb (with its particle) and the
object of S1 and S3, with a list of opposite verbs that never match, and keep the runs where S2 lasts
less than 1 s. Then the hand timeline (P3) must show the same object in the same hand at the end of
S1 and the start of S3. The check never merges. A reviewer watches all of S2, because a brief real
action can sit inside it.

In [ ]:
RESULTS["row07"] = r = run_check(REGISTRY["row07"], ctx)
V.check_report(r)

#### Summary: pass counts, flagged segments, time and cost

A segment passes the section when no check flags it. A cut between two segments counts for both. A
segment is "pending" when it passed every check that could run, but at least one check waits for the
GPU primitives (P3, P4, P6).

Time is the wall time per unit on a fresh run: an API answer served from the cache adds back the
latency it had when it was first fetched. Totals add up the units. Row 2 and 3 runs its units in 8
parallel threads, so its elapsed time is lower than its total. API cost is the uncached price of
every call. Compute cost uses the assumed hourly rates (assumption 19). The last table projects the
cost to one hour of video.

In [ ]:
category_summary(["row05", "row02_03", "row04", "row06", "row07"], "8.1")

### 8.2 Segmentation errors / Granularity (sheet rows 8 to 17)

Granularity is the number of actions per segment: two actions sharing one segment, or one action cut
into several. Merging is the worse error, because a split can be fixed later and a merge cannot. So
row 8 is reviewed first.

Questions on one or two frames go to OpenAI Decisions. Questions on a clip (several frames from one
segment) go to Sol for row 8 and to OpenAI Decisions (gpt-6-luna) for rows 11, 12 and 14. The next
part shows why. Questions on caption text go to Jev. The frame signals from P3, P4 and P6 add evidence
where they ran.

#### Which model answers the clip questions

Rows 8, 11, 12 and 14 ask a closed question about a clip of up to 8 frames. Both candidate models ran
on exactly the same frames and questions: Sol (`gpt-6.1-sol`) and OpenAI Decisions (`gpt-6-luna`). The
table compares how often they agree, what they flag, and their cost and latency. The answers come from
the cache, so this comparison costs nothing to rerun.

In [ ]:
PROB = {"row08": "two_actions_prob", "row11": "fail_then_retry_prob", "row12": "continuous_prob",
        "row14": "two_or_more_prob"}

def run_with(key, model):
    saved = dict(cfg.clip_model)
    cfg.clip_model[key] = model
    try:
        return run_check(REGISTRY[key], ctx, save=False)
    finally:
        cfg.clip_model.clear()
        cfg.clip_model.update(saved)

CMP, rows = {}, []
for key, col in PROB.items():
    a, b = run_with(key, "sol"), run_with(key, "decisions")
    ea, eb = evidence(a.findings), evidence(b.findings)
    m = ea[["segment_id", "status", col]].merge(eb[["segment_id", "status", col]], on="segment_id",
                                                suffixes=("_sol", "_luna"))
    asked = m[m[f"{col}_sol"].notna() | m[f"{col}_luna"].notna()].copy()
    CMP[key] = asked
    lat = lambda r: r.cost_per_unit[r.cost_per_unit.api_calls > 0].seconds_uncached.median()
    rows.append({"check": key, "units asked": len(asked),
                 "Sol candidates": int((asked.status_sol == "candidate").sum()),
                 "luna candidates": int((asked.status_luna == "candidate").sum()),
                 "same status": round(float((asked.status_sol == asked.status_luna).mean()), 3),
                 "probability correlation": round(float(asked[f"{col}_sol"].corr(asked[f"{col}_luna"])), 2) if len(asked) > 2 else None,
                 "Sol USD": round(a.loaded.check_usd.sum(), 4), "luna USD": round(b.loaded.check_usd.sum(), 4),
                 "Sol median s per call": round(lat(a), 2), "luna median s per call": round(lat(b), 2)})
pd.DataFrame(rows).set_index("check")

**Row 8 has a near ground truth.** By the guideline, "pick up X then put X" is two actions, and the caption
itself says so. Sol scores most of these segments high. Luna scores most of them near zero, so it would
miss merges, the one error that cannot be fixed later. Row 8 stays on Sol.

In [ ]:
r8 = CMP["row08"].merge(prims.segments[["segment_id", "caption"]], on="segment_id")
r8[r8.caption.str.contains(" then ")][["segment_id", "caption", "two_actions_prob_sol", "two_actions_prob_luna"]]

**Rows 11, 12 and 14 have no ground truth.** Every segment where the two models disagree was reviewed
against the same 8 frames the models saw. The labels are in `evaluation/clip_model_review.csv`. They are
one reviewer's reading of 8 frames, so they are provisional, not gold.

In [ ]:
review = pd.read_csv(ROOT / "evaluation" / "clip_model_review.csv")
display(review.groupby("check").favours.value_counts().unstack(fill_value=0))
review[["check", "segment_id", "sol_prob", "luna_prob", "label", "favours", "note"]]

In [ ]:
seg_idx = prims.segments.set_index("segment_id")
for r in review.itertuples():
    s = seg_idx.loc[r.segment_id]
    start = seg_idx.loc[r.prev_id].start_s if isinstance(r.prev_id, str) else s.start_s
    V.clip_strip(prims, s.recording_id, start, s.end_s, n=8,
                 title=f'{r.check} {r.segment_id}: "{s.caption}" | Sol {r.sol_prob:.2f}, luna {r.luna_prob:.2f} | review: {r.label} ({r.favours})')

**Decision.** On the 14 disagreements the review favours luna 8 times and Sol 3 times, with 3 unclear.
Luna is also about 20 times cheaper and about 3 times faster per call. So rows 11, 12 and 14 use OpenAI
Decisions (`gpt-6-luna`) and row 8 uses Sol. The setting is `cfg.clip_model`. Revisit it on the gold set.

The review also found errors for other checks: two "Place the tomatoes" segments start while the person
is still washing (a late cut, rows 2 and 3), and another person's hands are in view in sample 3 around
293 to 297 s and 309 s (row 40).

#### Row 8: two actions merged into one segment

**Approach.**
1. Text mark: two action clauses joined by "then" (or by "and" with the same hand). Two clauses with
   different hands joined by "and" are the two hands exemption, so they are not marked.
2. Length mark: the segment is in the top 20 percent of length for its verb (P7).
3. Frame mark: a hand shows three or more contact changes (P3), when it ran.
4. A random 10 percent of short one clause segments goes to review, since a fluent caption can hide
   a second action.
5. Sol looks at up to 6 frames of each marked segment: does the clip show two separate actions with
   a clear moment where the next one starts?

A text mark is a candidate unless Sol says the clip is one continuous action. Then it is reported,
because the guideline's own towel caption has "then" but is one segment. Any other mark needs Sol at
0.7 or more.

In [ ]:
RESULTS["row08"] = r = run_check(REGISTRY["row08"], ctx)
V.check_report(r)
display(Markdown("Detail with the evidence behind each answer:"))
e8 = evidence(r.findings)
display(e8[e8.status.isin(["candidate", "report"])][["segment_id", "status", "message", "text_mark", "long_mark",
                                                      "two_actions_prob"]])
show_clip(e8[e8.status == "candidate"].sort_values("two_actions_prob", ascending=False).segment_id.iloc[0])

#### Row 9: one action split into several segments

**Approach.** Mark a cut where both sides have the same verb and the same object, or the second caption
starts with "continue". OpenAI Decisions compares the frames 0.3 s before and after the cut: does one
action continue, or does a new one start? Flag when "one action" reaches 0.7. Contact does not decide
this row, because a new action can start without a release. This ranks below row 8: a split can be
fixed later.

In [ ]:
RESULTS["row09"] = r = run_check(REGISTRY["row09"], ctx)
V.check_report(r)
display(Markdown("Detail with the evidence behind each answer:"))
e9 = evidence(r.findings)
display(e9[e9.status != "pass"][["segment_id", "prev_id", "status", "message", "one_action_prob", "new_action_prob"]])

The flagged cuts in sample 2 sit between captions with the same text ("Pick up a small bag of black
parts ... then put the small bag in the large clear bag"). The frames show both hands working the bags
continuously, so a reviewer must decide at the native frame rate whether each cut splits one action or
separates two cycles. That is the intended use of a candidate.

#### Row 10: a micromovement folded into the next action

**Approach.** A folded micromovement has no caption, so no text or frame check can find it. It shows
only against a gold set: a gold segment with no annotator boundary within 0.2 s of its start. The check
reads `gold_segments.csv` from the data folder when it exists. There is none yet, so the result is
"no coverage", never a pass.

In [ ]:
RESULTS["row10"] = r = run_check(REGISTRY["row10"], ctx)
V.check_report(r)

#### Row 11: a failed and a successful attempt left in one segment

**Approach.** Take hand action segments longer than the median for their verb, every caption with a
failure word, and a random 10 percent of the rest (a short failure can hide in a short segment). OpenAI
Decisions looks at up to 8 frames: does the clip show a failed attempt and then a new attempt? Repeated small
adjustments that gradually work an object into place count as one attempt, as in the guideline's towel
example. The model can say yes or no but cannot place the split frame at 2 frames per second. The hand
timeline (P3) places it when it ran.

In [ ]:
RESULTS["row11"] = r = run_check(REGISTRY["row11"], ctx)
V.check_report(r)
display(Markdown("Detail with the evidence behind each answer:"))
e11 = evidence(r.findings)
display(e11[e11.status != "pass"][["segment_id", "status", "message", "failure_word", "long_mark", "fail_then_retry_prob"]])
for sid in e11[e11.status == "candidate"].segment_id.head(2):
    show_clip(sid)

#### Row 12: attempts split although they run as one continuous motion

**Approach.** Mark a cut where both sides act on the same object and either caption has an attempt or
failure word. OpenAI Decisions looks at both segments together: do the attempts run as one continuous
motion with no frame where a new attempt begins? Flag at 0.7 or more. An answer between 0.3 and 0.7 is kept as unsure.

In [ ]:
RESULTS["row12"] = r = run_check(REGISTRY["row12"], ctx)
V.check_report(r)
display(Markdown("Detail with the evidence behind each answer:"))
e12 = evidence(r.findings)
display(e12[e12.status != "pass"][["segment_id", "prev_id", "status", "message", "continuous_prob"]])

#### Row 13: an exemption split anyway

**Approach.** The guideline keeps three cases in one segment: walking with a hand action, both hands at
once, and sitting. Mark a cut between a walk and a hand action, or between hand actions with different
hands. Then compare the time spans of the actions from the motion timeline (P4, forward movement) and
the hand timeline (P3, contact). If the spans overlap by more than 0.3 s, the split was wrong. The
segments touch, so the gap between them is always zero; only the action spans can show the overlap.
Without P3 and P4 a marked cut is unassessable.

In [ ]:
RESULTS["row13"] = r = run_check(REGISTRY["row13"], ctx)
V.check_report(r)

#### Row 14: separate pick and place cycles grouped into one segment

**Approach.** The guideline counts three shakes of a blanket as one segment and four utensils loaded one
at a time as eight. Mark segments with a plural object (or "each", "all") or in the top 20 percent of
length for their verb. OpenAI Decisions counts the pick and place cycles in up to 8 frames. Flag when
two or more cycles reach 0.7. Repeats on one object without a place count as zero. Sol counted many
single pick and place segments as two cycles; see the comparison above.

In [ ]:
RESULTS["row14"] = r = run_check(REGISTRY["row14"], ctx)
V.check_report(r)
display(Markdown("Detail with the evidence behind each answer:"))
e14 = evidence(r.findings)
display(e14[e14.status != "pass"][["segment_id", "status", "message", "plural_object", "long_mark", "cycles_answer"]])

#### Row 15: a regrip or small adjustment cut as an action

**Approach.** Mark a segment under 0.5 s, a caption with an adjustment verb (adjust, regrip, shift), or
a segment with the same verb and object as a neighbour. Each mark is a hint only: a regrip can lose
contact for a moment and can last longer than 0.5 s, so no rule defines it. A seeded 20 percent of the
flags goes to review. The hand timeline adds contact changes around the segment when it ran.

In [ ]:
RESULTS["row15"] = r = run_check(REGISTRY["row15"], ctx)
V.check_report(r)

#### Row 16: a hand to hand transfer outside a walk, not recorded

**Approach.** Mark a cut where both sides act on the same object, the hand changes from left to right
or back, neither side is a walk, and neither caption says "transfer". The guideline shows a transfer
inside a walk caption on pages 3 and 9, so walks are skipped. OpenAI Decisions looks at the last frame
of the first segment and an early frame of the second: does the object move to the other hand? Flag at
0.7 or more. When the frames show the same hand, the caption hands may be wrong, so the cut is reported
for row 39 instead.

In [ ]:
RESULTS["row16"] = r = run_check(REGISTRY["row16"], ctx)
V.check_report(r)
display(Markdown("Detail with the evidence behind each answer:"))
e16 = evidence(r.findings)
display(e16[e16.status != "pass"][["segment_id", "prev_id", "status", "message", "moved_prob", "answer"]])

#### Row 17: an aborted reach cut before the hand withdraws

**Approach.** Jev reads every hand action caption: does it describe a reach or grasp that failed or was
pulled back? For each yes, the hand timeline (P3) checks the last 0.5 s: if the hand is still extended,
the segment ended before the withdrawal. The guideline defines the start of a reach but not the end of a
failed one, so this needs one line in the FAQ.

In [ ]:
RESULTS["row17"] = r = run_check(REGISTRY["row17"], ctx)
V.check_report(r)
display(Markdown("Detail with the evidence behind each answer:"))
e17 = evidence(r.findings)
display(e17.sort_values("failed_reach_prob", ascending=False).head(5)[["segment_id", "status", "message", "failed_reach_prob"]])

#### Summary: pass counts, flagged segments, time and cost

A segment passes the section when no check flags it. A cut between two segments counts for both. A
segment is "pending" when it passed every check that could run, but at least one check waits for the
GPU primitives (P3, P4, P6).

In [ ]:
category_summary(["row08", "row09", "row10", "row11", "row12", "row13", "row14", "row15", "row16", "row17"], "8.2")

### 8.3 Segmentation errors / Non-action states (sheet rows 18 to 27)

These checks cover stretches where the camera wearer is not doing the task: pauses (IDLE), sitting
(SITTING), and actions that are not part of the task (TASK UNRELATED ACTION). They look for a stretch
that is missed or mislabelled, or a fixed caption that covers a real action.

**Signals.** The hand timeline (P3) and the motion timeline (P4) give the signals when they ran. Without
them, OpenAI Decisions answers closed questions on sampled frames. Two questions are shared: the posture
of the camera wearer (rows 20 to 22) and what the person is doing (rows 23 and 24). Each is sent in
exactly the same form, so the second check reads the cached answer at no extra cost.

**What did not work.** A cheap signal without a GPU, the pixel difference between consecutive frames,
was tested as a stand-in for "the person is still". It does not separate pauses from actions: in samples
1 and 2 the IDLE segments change as much from frame to frame as the action segments, because the head
keeps moving while the hands rest. It is not used.

#### Row 18: a pause of 1 s or more left inside an action segment

**Approach.** For every action segment longer than 1 s: with P3 and P4, find stretches of 1 s or more
where both hands are seen with no contact, no hand motion and no walking. A hand that is not seen is
unknown, so it never counts as still. Without them, OpenAI Decisions looks at up to 12 frames and answers
whether there is a pause of about a second where the hands are idle (looking around counts as a pause,
since the guideline calls it IDLE). Flag at 0.7 or more. The reviewer labels the stretch as IDLE,
navigation or action.

In [ ]:
RESULTS["row18"] = r = run_check(REGISTRY["row18"], ctx)
V.check_report(r)
for sid in r.findings[r.findings.status == "candidate"].segment_id.head(3):
    show_clip(sid, n=10)

#### Row 19: an IDLE under 1 s

**Approach.** The guideline allows a short IDLE (a pause under 1 s may be folded, so it may also stand
alone). This row reports the share and never scores it. The threshold stays at 1 s, including exactly
1.0 s; a duration within 0.1 s of it is noted, because the CSV rounds times to 0.1 s.

In [ ]:
RESULTS["row19"] = r = run_check(REGISTRY["row19"], ctx)
V.check_report(r)

#### Row 20: hand actions captioned while the person sits

**Approach.** Seated hand actions belong to the SITTING segment, so they must not be captioned. OpenAI
Decisions reads the posture from up to 3 frames of every captioned segment (standing, seated, kneeling or
squatting, unclear). Flag "seated" at 0.7 or more. Every recording is checked, including those with no
SITTING label, since a whole recording can miss SITTING. With P4, a downward camera move adds evidence.

In [ ]:
RESULTS["row20"] = r = run_check(REGISTRY["row20"], ctx)
V.check_report(r)
e20 = evidence(r.findings)
display(e20.posture.value_counts().rename("segments by posture"))

#### Row 21: SITTING that does not cover sitting down or standing up

**Approach.** SITTING includes the act of sitting down and of standing up. OpenAI Decisions reads the
posture in the frame 0.3 s before the SITTING segment and 0.3 s after it. Already seated or sitting down
before it, or still seated or standing up after it, means the transition sits outside the segment. The
samples have no SITTING segment, so there is nothing to check.

In [ ]:
RESULTS["row21"] = r = run_check(REGISTRY["row21"], ctx)
V.check_report(r)

#### Row 22: kneeling or squatting

**Approach.** The guideline defines only SITTING. Kneeling at a low shelf or squatting at a dishwasher has
no rule. This row reads the same posture answer as row 20 and sends every "kneeling or squatting" to a
policy queue for the client. It is never scored and never penalises the annotator.

In [ ]:
RESULTS["row22"] = r = run_check(REGISTRY["row22"], ctx)
V.check_report(r)

#### Rows 23 and 24: what happens inside an IDLE segment

IDLE covers two cases: the person does nothing, or the reviewer cannot tell what the person does. Both
rows read one answer from OpenAI Decisions on 3 frames from the middle half of the IDLE segment. The
frames at the edges show the neighbouring actions and pushed the answer toward "task action", so they
are left out. Hands that rest on an object without moving it count as not acting.

**Row 23 approach.** For IDLE longer than 3 s: flag a clear task unrelated action (waving or talking to
someone, adjusting a headset, preparing the scene) at 0.7 or more. An answer between 0.3 and 0.7 is an
unclear case and is reported, not scored. The guideline prefers IDLE only when the case is unclear.

**Row 24 approach.** For every IDLE: flag when the frames show a task action or walking at 0.7 or more.
With P3 and P4, contact, hand motion and forward movement add evidence. The reviewer decides whether the
action is identifiable and writes the caption.

In [ ]:
RESULTS["row23"] = r = run_check(REGISTRY["row23"], ctx)
V.check_report(r)
RESULTS["row24"] = r = run_check(REGISTRY["row24"], ctx)
V.check_report(r)
for sid in r.findings[r.findings.status == "candidate"].segment_id:
    show_clip(sid, n=8)
idle_share = (prims.segments.assign(idle=prims.segments.caption.eq("IDLE"), d=prims.segments.end_s - prims.segments.start_s)
              .groupby("recording_id").apply(lambda g: g[g.idle].d.sum() / g.d.sum()).rename("IDLE share of time"))
idle_share.round(3)

#### Row 25: a task action marked TASK UNRELATED ACTION

**Approach.** There is no task description in the inputs, so Sol infers the task from all captions of the
recording, with a confidence. A low confidence makes the segment unassessable, never a pass. OpenAI
Decisions then asks whether the frames show a step of that task. The guideline lists scene preparation as
unrelated, and that list overrides task relevance. The samples have no TASK UNRELATED ACTION, so there is
nothing to check and no API call is made.

In [ ]:
RESULTS["row25"] = r = run_check(REGISTRY["row25"], ctx)
V.check_report(r)

#### Row 26: looking around confused with turning

**Approach.** Needs the motion timeline (P4): rotation with no forward movement is looking around or a
turn in place. Flag IDLE with sustained rotation, and a turn caption with almost no rotation, for review
only. A head turn and a body turn look the same on a head camera, so this needs a calibration round with
three reviewers first. Without P4 every unit is unassessable.

In [ ]:
RESULTS["row26"] = r = run_check(REGISTRY["row26"], ctx)
V.check_report(r)

#### Row 27: a non-action stretch missed inside a long segment

**Approach.** Take segments in the top 10 percent of length for their verb (and longer than 2 s). Read
rows 18 and 20 for them: a pause is a missed IDLE, sitting is a missed SITTING. A duration screen misses
short special states and recordings with no special label at all, so a random 10 percent of whole
recordings also goes to a full human audit.

In [ ]:
RESULTS["row27"] = r = run_check(REGISTRY["row27"], ctx)
V.check_report(r)
from bishop_qc.checks.nonaction import audit_recordings
print("Recordings picked for a full human audit:", ", ".join(audit_recordings(ctx)))

#### Summary: pass counts, flagged segments, time and cost

A segment passes the section when no check flags it. A segment is "pending" when it passed every check
that could run, but at least one check waits for the GPU primitives (P3, P4, P6).

In [ ]:
category_summary(["row18", "row19", "row20", "row21", "row22", "row23", "row24", "row25", "row26", "row27"], "8.3")

### 8.4 Caption errors / Faithfulness to what happened (sheet rows 28 to 34)

The caption must say what the camera shows: what happened, not what was meant, and nothing the camera
did not see. Object and place questions use SAM 3 (P6) when it ran. Without it, OpenAI Decisions answers
closed questions on sampled frames. Caption text goes to Jev. Sol is used only for the last tier of row 33.

**Precision is not yet known for this category.** The frames are 640 pixels wide, so a fork or a sponge
can be a few pixels across, and a model can miss it. A missed object is a candidate for review, never a
verdict. These checks surface candidates; their thresholds must be set on a gold set.

#### Row 29: a failure softened or left out

**Approach.** The caption hides the failure, so the check starts from the frames. For every hand action
whose caption records no failure or attempt, OpenAI Decisions looks at 6 frames: does anything go wrong
(a miss, a slip, a drop, a spill, or a reach pulled back)? Flag at 0.7 or more. With P3 and P6, a release
before the end with no place verb, or an object that falls, adds evidence. A caption accepted from a
suggestion without edits is noted, because suggestions describe clean actions and are weakest on failures.

In [ ]:
RESULTS["row29"] = r = run_check(REGISTRY["row29"], ctx)
V.check_report(r)

#### Row 30: intent wording with no stated outcome

**Approach.** Search for intent phrases ("try to", "attempt to", "start to"). For each hit, Jev answers
whether the caption states what happened. An outcome can follow "but" or "then", or be the main verb
("Fail to open the jar"). Flag an intent phrase with no outcome. Sample 2 row 26 ("Attempt to push ... but
fail to join them") states its outcome, so it passes.

In [ ]:
RESULTS["row30"] = r = run_check(REGISTRY["row30"], ctx)
V.check_report(r)

#### Row 31: an object named but not visible

**Approach.** Take every object the caption names (not pronouns or body parts). With SAM 3 (P6), the
object is searched for in the segment's frames. Without it, OpenAI Decisions answers one question per
object on 5 frames spread over the segment: is it visible in at least one? Flag when the probability of
"visible" is 0.3 or less. The check never proposes a new noun. A reviewer picks one outcome: visible, not
visible, or visible only outside the segment (the guideline conflict on held objects).

In [ ]:
RESULTS["row31"] = r = run_check(REGISTRY["row31"], ctx)
V.check_report(r)

#### Row 32: a destination not visible, or the object does not end up there

**Approach.** Only for verbs that move an object to a place (put, place, insert, drop, move, ...). "Rinse
the fork under the tap" names where the action happens, not a destination, so it is skipped. OpenAI
Decisions looks at the last second of the segment: is the place visible, and is the object in or on it in
the last frame? Flag either at 0.3 or less. If the place is out of view, the caption should stop at what is
visible (guideline page 7).

In [ ]:
RESULTS["row32"] = r = run_check(REGISTRY["row32"], ctx)
V.check_report(r)

#### Row 33: the verb is wrong, or the described action did not happen

**Approach.** Three tiers. Most segments stop at tier 1 or 2.
1. **Tier 1 (P3).** The verb against the contact pattern: pick up means contact starts, place or put means
   contact ends.
2. **Tier 2 (OpenAI Decisions).** The first and the last frame against the start and end of the caption.
3. **Tier 3 (Sol).** Only for segments that tier 1 or 2 marks, plus a random 5 percent: Sol looks at 8
   frames and says whether the caption is correct, and if not, what is wrong (wrong action, wrong object,
   wrong hand, the described result never happens, or extra actions in the clip).

A "correct" probability under 0.7 is a candidate with its reason. "The clip also shows another action" is
reported instead, because the caption is right and the extra action is a boundary or merge question for
rows 2, 3 and 8. Every Sol answer must be measured against reviewers on a gold set before its pass rate is
trusted.

In [ ]:
RESULTS["row33"] = r = run_check(REGISTRY["row33"], ctx)
V.check_report(r)
e33 = evidence(r.findings)
display(e33.tier3_reason.fillna("not sent to tier 3").value_counts().rename("segments by tier 3 answer"))
for sid in e33[e33.status == "candidate"].sort_values("tier3_correct_prob").segment_id.head(3):
    show_clip(sid, n=8)

#### Row 34: a confident guess where a fallback noun or IDLE was needed

**Approach.** For every hand action with a specific object noun (not a fallback noun), take the sharpest
frame of the segment (P2 blur score). OpenAI Decisions answers whether a person can tell the object is what
the caption says: identifiable, not identifiable, clearly something else, or unclear. Flag "not
identifiable" at 0.7 or more. This check is weak by nature: a correct guess looks the same as knowledge.

In [ ]:
RESULTS["row34"] = r = run_check(REGISTRY["row34"], ctx)
V.check_report(r)
e34 = evidence(r.findings)
display(e34.answer.value_counts().rename("segments by answer"))

#### Row 28: the caption describes what was meant, not what happened

**Approach.** No separate model. Row 28 collects the intent wording without an outcome from row 30, and the
row 33 cases where Sol says the described result never happens. A random 5 percent of captions also goes
to review against the video.

In [ ]:
RESULTS["row28"] = r = run_check(REGISTRY["row28"], ctx)
V.check_report(r)
print("Captions in the 5 percent review sample:", int(evidence(r.findings).review_sample.sum()))

#### Summary: pass counts, flagged segments, time and cost

In [ ]:
category_summary(["row29", "row30", "row31", "row32", "row33", "row34", "row28"], "8.4")

### 8.5 Caption errors / Completeness of the instruction (sheet rows 35 to 37)

A caption is an instruction a robot could carry out without the video. These checks look for a missing
element: the object, the hand, the action, the place, or a stopping point. The field checks read the
caption fields (P5) and need no model, so they are cheap and exact. Frames are read only to decide
whether a place or an end state was visible.

#### Row 35: a required element is missing

**Approach.** Check each action clause, not the caption as a whole.
1. A hand action needs a verb, an object and an actor (a hand or a body part). An actor can be shared
   inside one sentence ("Pick up the cup and place it with your right hand"). The outcome of an attempt
   ("... but fail to join them") shares the attempt's hand.
2. Navigation needs a verb and a landmark (or what the person ends up facing).
3. A clause that moves an object somewhere (put, place, move, drop, ...) but names no place: OpenAI
   Decisions looks at the last frame and answers whether the object now rests on a place you can see and
   name. If yes, the place is missing. A place out of view is exempt, as the guideline asks the caption to
   stop at what is visible. A direction ("to the right") is not a place.

The fixed captions are exempt. A clause with no hand also counts for row 38 (hands); the two rows report
the same gap from two angles.

In [ ]:
RESULTS["row35"] = r = run_check(REGISTRY["row35"], ctx)
V.check_report(r)

#### Row 36: an action with a defined end point does not state it

**Approach.** Only clauses whose verb has a defined end point (close, open, twist, fill, fold, seal,
turn on, ...; the list needs client sign off). Jev answers whether the caption states the end point. A
verb such as close already states it, and "until it is closed" is one way, not the only way. If Jev says
no, OpenAI Decisions checks the last frame: has the object reached a clear end state? Flag only when it
has. An end point the frames do not show is never inferred.

In [ ]:
RESULTS["row36"] = r = run_check(REGISTRY["row36"], ctx)
V.check_report(r)

#### Row 37: an action that briefly leaves the view is not described

**Approach.** The guideline asks for an action that passes out of view to be described when it is
unmistakable from the frames either side (a fork leaves the frame in one hand and comes back in the
other). Take IDLE segments, and segments under 1 s next to a hand action. With P3 and P6, the hand
timeline and the object tracks compare what each hand holds at the start and the end. Without them,
OpenAI Decisions compares the first and last frames: did an object move to the other hand, leave a hand,
or appear in one? Flag an IDLE segment with such a change at 0.7 or more. A disappearance alone never says
what happened, so the reviewer decides whether it was unmistakable.

In [ ]:
RESULTS["row37"] = r = run_check(REGISTRY["row37"], ctx)
V.check_report(r)

#### Summary: pass counts, flagged segments, time and cost

In [ ]:
category_summary(["row35", "row36", "row37"], "8.5")

### 8.6 Caption errors / Hands and body attribution (sheet rows 38 to 49)

Every hand action names its hand, the right one, and what each hand holds when it matters. Text checks read
the caption fields and need no model. Frame checks use the hand timeline (P3) and object tracks (P6) when
they ran, otherwise OpenAI Decisions on sampled frames. Every hand question tells the model that the camera
is on the person's head, so their left hand appears on the left of the image.

#### Row 38: the acting hand is not named

**Approach.** Text only. Every action clause needs a hand or body part. A hand can be shared inside one
sentence ("Pick up the cup and place it with your right hand"), and the outcome of an attempt ("... but fail
to join them") shares the attempt's hand. A reach without contact is still a hand action. Navigation clauses
are exempt, since idle hands are not mentioned. Row 35 reports the same gap as a missing element.

In [ ]:
RESULTS["row38"] = r = run_check(REGISTRY["row38"], ctx)
V.check_report(r)

#### Row 39: the wrong hand is named

**Approach.** For each hand action naming left, right or both hands: with P3 and P6, the hand in contact
with the object's box during the action, compared per action rather than by a vote over the whole segment.
Without them, OpenAI Decisions looks at 5 frames and answers which hand does the action on the object. Flag
when another answer reaches 0.7. Crossed hands and turned heads confuse hand side, so those cases are
oversampled for review. Another person's hands are handled first (row 40).

In [ ]:
RESULTS["row39"] = r = run_check(REGISTRY["row39"], ctx)
V.check_report(r)
for sid in r.findings[r.findings.status == "candidate"].segment_id.head(2):
    show_clip(sid, n=6)

#### Row 40: another person's hand in view

**Approach.** The guideline never captions other people's hands, but nothing in it covers them in view. With
P3, a hand box that does not reach the bottom edge of the frame is marked "unclear owner"; with P6, SAM 3
looks for "person". Without them, OpenAI Decisions answers whether a hand or arm of another person is visible
in 3 frames. Each signal only routes the segment to a reviewer, who checks that the caption describes only
the camera wearer's hands.

In [ ]:
RESULTS["row40"] = r = run_check(REGISTRY["row40"], ctx)
V.check_report(r)

#### Row 41: an idle hand is mentioned

**Approach.** Two cases. A phrase that describes an idle hand ("hands at your sides", "left hand is idle")
is flagged from the text. A "while holding X in your Y hand" clause is checked against the frames: if that
hand does not hold the object, the caption names a hand that is idle, or the hands are swapped (row 39).

In [ ]:
RESULTS["row41"] = r = run_check(REGISTRY["row41"], ctx)
V.check_report(r)

#### Row 42: the other hand's action is left out

**Approach.** For hand actions whose caption names one hand only (and no held object): does the other hand
do an action of its own, picking an object up, putting one down, or moving or working on a different object?
Holding something still, resting, or settling onto an object the first hand holds does not count; whether
held contents must be named is rows 45 and 46.

**What changed.** A first version also counted "holding it for the action". It flagged 59 percent of
segments, because in sample 2 the left hand holds the large bag throughout. Narrowed to an action of its own,
it flags 5 of 73.

In [ ]:
RESULTS["row42"] = r = run_check(REGISTRY["row42"], ctx)
V.check_report(r)

#### Rows 43 and 44: a transfer between hands

**Row 43 approach.** The caption side of row 16: the same object moves from one hand to the other across a
cut, outside a walk, with no transfer recorded. It sends the same question as row 16, so the answer comes
from the cache. When the frames show the same hand, the reviewer decides between a missing transfer and a
wrong hand (row 39).

**Row 44 approach.** For row 43's cases: is the moment of the transfer visible in the frames around the
cut? If not, the transfer happened out of view. The guideline still asks for it to be described when it is
unmistakable from the frames either side, and the reviewer judges that.

In [ ]:
RESULTS["row43"] = r = run_check(REGISTRY["row43"], ctx)
V.check_report(r)
RESULTS["row44"] = r = run_check(REGISTRY["row44"], ctx)
V.check_report(r)

#### Row 45: the carried object is not named during navigation

**Approach.** For every navigation segment: does the person carry an object, and in which hand? With P3 and
P6 the carried objects and hands are read from the timelines; without them OpenAI Decisions answers on 4
frames. Flag a carried object when the caption has no "while holding" clause, since leaving it out implies
empty hands. The samples have no navigation segment.

In [ ]:
RESULTS["row45"] = r = run_check(REGISTRY["row45"], ctx)
V.check_report(r)

#### Row 46: the other hand's contents listed when they do not bear on the action

**Approach.** For hand actions with a "while holding" clause, Jev answers whether the held object matters to
the main action. A low answer is reported, never scored: this is minor and a judgement, and the guideline's
own fork example lists such contents.

In [ ]:
RESULTS["row46"] = r = run_check(REGISTRY["row46"], ctx)
V.check_report(r)

#### Rows 47 and 48: body parts other than hands

**Row 47 approach.** For actions on a door, drawer, lid, cabinet or similar: what moves it, a hand, the hip,
a foot, a knee or an elbow? "Unknown" is kept as an answer, since another person, gravity or camera motion can
also explain the movement. Flag a body part at 0.7 or more when the caption names none.

**Row 48 approach.** Every caption that names a body part goes to review, because full hands and a moving door
do not prove a hip push. A hand seen touching the object while it moves makes the claim doubtful.

In [ ]:
RESULTS["row47"] = r = run_check(REGISTRY["row47"], ctx)
V.check_report(r)
RESULTS["row48"] = r = run_check(REGISTRY["row48"], ctx)
V.check_report(r)

#### Row 49: grip, fingers or arm path described

**Approach.** Search for grip words (fingers, thumb, grip, palm, tightly, ...). Jev answers whether the word
describes how the hand holds, rather than naming an object or the action itself ("pinch" is a real verb in
the guideline's towel example). A grip description is a warning; the fix deletes the phrase in a proposed
edit and the original is kept.

In [ ]:
RESULTS["row49"] = r = run_check(REGISTRY["row49"], ctx)
V.check_report(r)

#### Summary: pass counts, flagged segments, time and cost

In [ ]:
category_summary(["row38", "row39", "row40", "row41", "row42", "row43", "row44", "row45", "row46", "row47",
                  "row48", "row49"], "8.6")

### 8.7 Caption errors / Navigation (sheet rows 50 to 57)

Movement is described so a reader can trace the route: against fixed landmarks, with the direction of a turn
and what the person ends up facing. Only clauses where the camera wearer moves count. "Turn on the tap" and
"Move the fork to the right" move an object, not the person. The parser decides first, and Jev confirms each
clause that has a navigation verb.

| Row | Check | Approach |
|---|---|---|
| 50 | No landmark | Each movement needs a landmark ("toward the doorway", "past the island") or what the person faces. A turn followed by a walk with a landmark in the same caption is covered by the walk. |
| 51 | Bare direction sequence | Remove the verbs, the connectors and the direction words. Nothing left means a bare sequence like "go forward then turn right". |
| 52 | Landmark not static | Fixtures and the guideline's own furniture pass from a list. Any other landmark goes to Jev: fixture, large furniture, small object or person. A small object or a person is flagged. |
| 53 | Furniture outside the examples | The guideline uses a sofa, a table and a shelf. Other furniture (a chair, a stool) is reported and counted for the client, never scored. |
| 54 | Navigating by the target object | The landmark is the object of the next action, in the same caption or the next segment. A fixture is exempt, since heading toward a sink still qualifies. |
| 55 | Turn without direction or end target | A turn needs left or right, and what the person ends up facing. With the motion timeline (P4), a walk with sustained camera rotation but no turn in the caption is flagged too. |
| 56 | "Turn around" | Flagged every time. The fix names the direction and the end target. |
| 57 | Route not traceable | For walks over 3 s, OpenAI Decisions looks at 8 frames: does the route pass fixtures the caption does not name? Review only: not every visible landmark must be named. |

In [ ]:
from bishop_qc.checks.navigation import NAVIGATION_CHECKS

for key in NAVIGATION_CHECKS:
    RESULTS[key] = r = run_check(REGISTRY[key], ctx)
    V.check_report(r)

The sample recordings have no walking or turning captions. Every segment above with a navigation verb moves
an object ("Move the fork to the right", "Turn on the tap"), so it correctly passes.

#### Navigation test set

To show the checks working, the next cell defines a small set of made-up captions. Each one lists the rows
it should flag and why. They form one test recording in this order, so row 54 can read the next caption.
There is no video for them, so the frame part of row 57 does not apply (every test walk is under 3 s).

In [ ]:
NAV_TEST = [
    {"caption": "Walk toward the doorway", "expect": "", "why": "A fixture landmark. Passes."},
    {"caption": "Walk forward", "expect": "50, 51", "why": "No landmark, and only a verb and a direction."},
    {"caption": "Go forward then turn right", "expect": "50, 51, 55", "why": "A bare direction sequence; the turn has no end target."},
    {"caption": "Turn right until you are facing the sink", "expect": "", "why": "Direction and end target. Passes."},
    {"caption": "Turn toward the fridge", "expect": "55", "why": "End target but no direction."},
    {"caption": "Turn around", "expect": "50, 51, 55, 56", "why": "'Turn around' is banned; no direction, target or landmark."},
    {"caption": "Turn right and walk past the kitchen island toward the sink", "expect": "", "why": "The guideline's own example. Passes."},
    {"caption": "Walk past the chair toward the sink", "expect": "53", "why": "A chair is furniture outside the guideline's examples: reported for the client."},
    {"caption": "Walk toward the person", "expect": "52", "why": "A person is not a static landmark."},
    {"caption": "Walk toward the cup", "expect": "52, 54", "why": "A cup is not static, and it is the next segment's object."},
    {"caption": "Pick up the cup from the table with your right hand", "expect": "", "why": "Not navigation."},
    {"caption": "Walk toward the table", "expect": "", "why": "The next object sits on the table, but the table is a fixture landmark."},
    {"caption": "Pick up the cup from the table with your right hand", "expect": "", "why": "Not navigation."},
    {"caption": "Turn on the tap with your right hand", "expect": "", "why": "Turns the tap, not the person. Must not flag."},
    {"caption": "Move the fork to the right with your right hand", "expect": "", "why": "Moves an object, not the person. Must not flag here."},
    {"caption": "Walk toward the sofa while holding the cup in your right hand", "expect": "", "why": "The guideline's own example. Passes."},
]
pd.DataFrame(NAV_TEST)

In [ ]:
from bishop_qc.checks.testset import caption_context, run_cases, expected_vs_actual

test_ctx = caption_context(cfg, clients, NAV_TEST, "nav_test")
test_results = run_cases(test_ctx, [REGISTRY[k] for k in NAVIGATION_CHECKS], NAV_TEST)
nav_check = expected_vs_actual(test_results, test_ctx, NAV_TEST)
display(Markdown(f"**{int(nav_check.match.sum())} of {len(nav_check)} test captions flagged exactly the expected rows.**"))
nav_check

#### Summary: pass counts, flagged segments, time and cost (sample data)

In [ ]:
category_summary(NAVIGATION_CHECKS, "8.7")

### 8.8 Caption errors / Object reference (sheet rows 58 to 71)

The guideline asks for the minimal description that makes the reference unique, plus at most one extra
attribute. Objects are named as seen, never from a label, a container's hidden contents or an earlier
segment. When an object cannot be identified, the caption uses one of four fallback nouns (garment,
container, utensil, linen) with an attribute the footage supports.

Text rules run first and are cheap. Jev answers questions about the caption text. OpenAI Decisions answers
questions about the frames: on a SAM 3 crop of the object when P6 ran, otherwise on a whole frame with the
object named. Sol suggests a specific name for a vague or fallback noun, once per recording and phrase, so the
reviewer sees a proposal. No check renames an object. Every frame answer is a candidate for review.

| Row | Check | Approach |
|---|---|---|
| 58 | Wrong object named | Nouns in a list of pairs people confuse (bowl and plate, cup and glass, lid and plate). On the sharpest frame, a closed choice between the caption's noun and its pair. Blurred frames go to review, not to the model. |
| 59 | Not specific enough | A vague head noun (thing, item, part, piece) is flagged by text. Sol's suggested name goes in the reason. |
| 60 | Does not single out one object | Two frames, at the start and the middle: can a reader tell which object the caption means? With SAM 3, the count of instances that match the phrase in one frame. Plural nouns are skipped. |
| 61, 62 | Needs an earlier segment | Context words (it, back, again, another, the last). Jev decides if each one points outside the caption. "Until it is closed" refers inside the caption and passes. Rows 61 and 62 give one finding. |
| 63 | Named from a label | Jev marks captions that name a brand, a product or label-only contents. Those go to the frames: is the name printed on the object? |
| 64 | Contents assumed | A container noun with an "of" phrase ("bag of black parts"). On the frame: can a person see the contents? |
| 65 | Fallback on an identifiable object | Five frames per fallback noun. Flag when three or more say a specific kind is clear, with Sol's suggested name. |
| 66 | Synonym for a fallback noun | Fabric, vessel, implement and similar words are flagged by text. The frame answer picks the fix: the specific name, or the fallback noun. |
| 67 | Fallback with no attribute | No attribute in the noun phrase, then Jev checks for one after the noun ("on the left"). The place it is taken from does not count. |
| 68 | Earlier caption edited later | P9 backward edits: the caption was updated after later captions were created. The reason adds whether the noun changed from the suggestion and whether it also appears in those later captions. |
| 69 | No fallback noun fits | Vague nouns only. On the frame: identifiable, one of the four fallback nouns, or none fits. "None fits" is reported for the client, never scored. |
| 70 | Over-detailed | Two or more attributes. The minimum is 0 when one object of the kind is in view and no other of the kind is named in the recording, otherwise 1. Flag above the minimum plus one. Minor. |
| 71 | Dialect words | Counts per recording from a map of pairs (tap and faucet). Reported, not scored: the guideline sets no dialect. |

In [ ]:
from bishop_qc.checks.objectref import OBJECT_REFERENCE_CHECKS

for key in OBJECT_REFERENCE_CHECKS:
    RESULTS[key] = r = run_check(REGISTRY[key], ctx)
    V.check_report(r)

**Reading the results on the samples.**

- Row 58 flags one "metal plate" as a possible lid. The frame shows a rimmed steel plate of the kind often
  used as a pot lid. That is a fair review case, not a clear error.
- Row 60 is unsure on identical, interchangeable items. On "Pick up a breaded piece from the white plate",
  the chance of "ambiguous" ranges from 0.19 to 0.59, below the 0.7 flag line. The guideline says to tell
  three cups apart by colour, size or position. It does not say whether that applies when any of the
  pieces would do. This is a question for the client.
- Row 68 finds the planted backward edit on `sample1_009`.
- Row 69 reports the white plastic parts in sample 2. No fallback noun fits a part of a gadget.

#### Object reference test set

The samples have no labels, synonyms or context words that point outside the caption. The next cell
defines made-up captions that do. Each one lists the rows it should flag and why. There is no video for
them, so the frame checks report "could not be assessed". This set tests the text side of rows 59, 61,
63, 66 and 67. The frame side is shown on the samples above.

In [ ]:
OBJ_TEST = [
    {"caption": "Pick up the thing from the table with your right hand", "expect": "59",
     "why": '"thing" is a vague noun'},
    {"caption": "Pick up the blue cup from the table with your right hand", "expect": "",
     "why": "specific noun, one attribute"},
    {"caption": "Put it back on the table with your right hand", "expect": "61",
     "why": '"it" and "back" need the earlier segment'},
    {"caption": "Pick up another blue cup from the shelf with your left hand", "expect": "61",
     "why": '"another" points to a cup in an earlier segment'},
    {"caption": "Twist the cap onto the bottle with your right hand until it is closed", "expect": "",
     "why": 'guideline example: "it" refers inside the caption'},
    {"caption": "Wash the cup again with both hands", "expect": "61",
     "why": '"again" needs the earlier segment'},
    {"caption": "Pick up the spinach bag from the counter with your left hand", "expect": "63",
     "why": "guideline example: names the bag by its printed label"},
    {"caption": "Pick up the Heinz bottle with your right hand", "expect": "63",
     "why": "brand name read off the label"},
    {"caption": "Pick up the leftmost white bag from the counter with your left hand", "expect": "",
     "why": "guideline's corrected version: described as seen"},
    {"caption": "Fold the fabric on the bed with both hands", "expect": "66",
     "why": '"fabric" in place of the fallback noun "garment"'},
    {"caption": "Put the vessel in the sink with your right hand", "expect": "66",
     "why": '"vessel" in place of the fallback noun "container"'},
    {"caption": "Pick up the container from the counter with your right hand", "expect": "67",
     "why": "fallback noun with no attribute"},
    {"caption": "Pick up the container on the left with your right hand", "expect": "",
     "why": "fallback noun with a position after it"},
    {"caption": "Pick up the gray garment from the couch with your left hand", "expect": "",
     "why": "guideline example: fallback noun with a colour"},
    {"caption": "Pick up the items from the table with both hands", "expect": "59",
     "why": '"items" is a vague noun'},
]
pd.DataFrame(OBJ_TEST)

In [ ]:
obj_ctx = caption_context(cfg, clients, OBJ_TEST, "obj_test")
obj_results = run_cases(obj_ctx, [REGISTRY[k] for k in OBJECT_REFERENCE_CHECKS], OBJ_TEST)
obj_check = expected_vs_actual(obj_results, obj_ctx, OBJ_TEST)
display(Markdown(f"**{int(obj_check.match.sum())} of {len(obj_check)} test captions flagged exactly the expected rows.**"))
obj_check

#### Summary: pass counts, flagged segments, time and cost (sample data)

In [ ]:
category_summary(OBJECT_REFERENCE_CHECKS, "8.8")